<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-12-production-deploy/lesson-12.5-ingestion/notebooks/GCP_Capstone_12.5_Ingestion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.5 Productionize Ingestion — Three Objects Into the Lane
**Netsetos GenAI Engineering — GCP Capstone** · Module 12 · rebuilt on the live lane, 10 September 2026

The one service nothing deployed until this lesson is the one that has been running the longest: `documind-ingest` indexed the fourteen real documents of every tenant on 6 September, and Module 9's figures and video segments after that. This notebook is where it comes from - the heredocs at the end are the kit's source - and the story now runs against it. A markdown note with a fact nobody else has goes in, the worker's one log line is waited for, the fact comes back through the one `retrieve()`. The same bytes go in again and are refused by the claim. A zero-byte object goes in and is refused at the edge with a 400, and the dead-letter subscription is peeked. A PNG goes in and becomes a figure chunk. The contracts, the claim race and the poison path stay as the theory they are, exercised against the lane's own modules.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-firestore==2.30.0 pydantic==2.13.5 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
sys.path.insert(0, f"{KIT}/deploy/services/ingest")   # contracts, idempotency: the worker's own modules, imported from the clone
from google.cloud import firestore
db = firestore.Client(project=PROJECT_ID)
import datetime
STAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")   # this run's objects and fact are unique

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120, token: str | None = "member") -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) - as documind-ui-sa by default, with token=None as
    nobody, or with a token minted as another account. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    headers = {}
    if token == "member":
        headers["Authorization"] = f"Bearer {documind_tools._id_token(API_URL)}"     # the audience is the canonical URL
    elif token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(f"{url}{path}", json=body, headers=headers, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def api_get(path: str, base: str | None = None, timeout: int = 60) -> tuple[int, dict | str]:
    """GET one API route as the roster member: /version, /health."""
    url = (base or API_URL).rstrip("/")
    r = requests.get(f"{url}{path}", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"), "sa": j["spec"]["template"]["spec"].get("serviceAccountName"),
            "annotations": (j["spec"]["template"]["metadata"].get("annotations") or {}),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query", service_name: str = "documind-api") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="{service_name}" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

# THE TWENTY LINES THAT MATTER, FROM THE CLONE. Module 12's notebooks are where the kit's files come from (the heredoc
# cells at the end of each notebook are what extract_documind.py reads), so the walls stay there and the story reads
# the file the lane actually runs, around one line, with the file's length beside it.
def excerpt(rel: str, needle: str, before: int = 0, after: int = 14) -> str:
    lines = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read().splitlines()
    i = next(n for n, l in enumerate(lines) if needle in l)
    lo, hi = max(0, i - before), min(len(lines), i + after)
    return f"# {rel}:{lo + 1}-{hi}  ({len(lines)} lines)\n" + "\n".join(lines[lo:hi])

def gcloud(*args: str) -> str:
    """One gcloud read, as the notebook's account, stdout only."""
    r = subprocess.run(["gcloud", *args, "--project", PROJECT_ID], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else f"(gcloud: {r.stderr.strip()[:200]})"

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # run_eval, usage_rows, judge
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules, for the excerpts and the pure functions
print("helpers: api(), api_get(), gateway(), service(), usage_rows(), excerpt(), gcloud(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: The lane, watched
A helper that waits for the worker's log line, and one that puts an object under the tenant's prefix.


In [ ]:
from contracts import IngestMessage, DocumentContract, sha256_of
from google.cloud.firestore_v1.base_query import FieldFilter

# THE LANE, WATCHED. An object under gs://PROJECT-uploads/<tenant>/ is the whole API of ingestion: Eventarc turns the
# finalize event into a Pub/Sub push to documind-ingest, the worker claims the content hash, parses, scans, chunks,
# embeds, indexes, and logs ONE line - ingest_ok with the tenant, the doc_key, the chunk and page counts. This helper
# waits for that line the way make ingest-one does, and the next cells upload three very different objects.
def wait_for(event: str, since: datetime.datetime, tenant: str | None = TENANT, minutes: int = 5) -> dict | None:
    """The worker's log line for this upload, or None: polls Cloud Logging every 10 s, up to `minutes`."""
    q = (f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-ingest" AND jsonPayload.event="{event}" '
         f'AND timestamp>="{since.strftime("%Y-%m-%dT%H:%M:%SZ")}"' + (f' AND jsonPayload.tenant="{tenant}"' if tenant else ""))
    for _ in range(minutes * 6):
        r = subprocess.run(["gcloud", "logging", "read", q, "--project", PROJECT_ID, "--limit", "1", "--format=json"], capture_output=True, text=True)
        rows = json.loads(r.stdout or "[]")
        if rows:
            return rows[0]["jsonPayload"]
        time.sleep(10)
    return None

def chunks_of(source_uri: str) -> list[dict]:
    """The chunks the worker wrote for one object - the two equality filters need no composite index."""
    q = (db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", TENANT))
           .where(filter=FieldFilter("source_uri", "==", source_uri)))
    return [d.to_dict() for d in q.stream()]

def upload(name: str, data: bytes, content_type: str) -> str:
    """One object under the tenant's prefix - the same gcloud storage cp the Makefile runs, as a client call."""
    blob = gcs.bucket(UPLOAD_BUCKET).blob(f"{TENANT}/{name}")
    blob.upload_from_string(data, content_type=content_type)
    return f"gs://{UPLOAD_BUCKET}/{TENANT}/{name}"

print("wait_for(event, since), upload(name, bytes, content_type) - the worker's contract:", excerpt("services/ingest/main.py", "def push(", 1, 3).splitlines()[0])


## Cell 3: A new document with a fact nobody else has


In [ ]:
# A NEW DOCUMENT WITH A FACT NOBODY ELSE HAS. Markdown is parsed directly (text/* skips Document AI), so this is the
# cheapest object the lane ingests - and the fact is unique to this run, so retrieving it afterwards proves the LANE
# indexed it, not that a demo corpus already had it. The doc_key is computed here the way the worker computes it
# (contracts.py: tenant + sha256 of the CONTENT), so the Firestore claim can be read back by name.
FACT_CODE = f"RN-{STAMP[-6:]}"
NOTE = f"""# Module 12 rehearsal note {STAMP}

The rehearsal reference code for this ingestion run is {FACT_CODE}. Quote it exactly when asked for the rehearsal reference code.

The note was uploaded from lesson 12.5 to show the lane ingesting a document end to end: the object landed in the uploads
bucket, Eventarc delivered it, the worker claimed its content hash, chunked it, scanned it, embedded it and indexed it.
""".encode()
since = datetime.datetime.now(datetime.timezone.utc)
uri = upload(f"module12_rehearsal_note_{STAMP}.md", NOTE, "text/markdown")
doc = DocumentContract(tenant_id=TENANT, sha256=sha256_of(NOTE), gcs_uri=uri, pages=0)
print("uploaded", uri, "| doc_key", doc.doc_key[:40], "...")
line = wait_for("ingest_ok", since)
assert line, "no ingest_ok in 5 min: gcloud logging read ... service_name=documind-ingest (is the push subscription alive? eventarc.tf)"
print("the worker's line:", {k: line.get(k) for k in ("event", "tenant", "doc_key", "chunks", "pages", "kinds")})
assert line["doc_key"] == doc.doc_key, "the worker's key is the content hash this notebook computed"
claim = db.collection("documents").document(doc.doc_key).get().to_dict()
print("the claim   :", {k: claim.get(k) for k in ("status", "chunks", "gcs_uri")})
assert claim["status"] == "indexed"


### Retrieved, through the one retrieve()


In [ ]:
# RETRIEVED, THROUGH THE ONE retrieve(). The fact is in the index a few seconds after the line; the kit's retrieve()
# (Modules 6-8: the single entry point every brain and both agents call) asks the API, and the API cites the chunk
# the worker wrote - kind text, the object's URI as the source. Nothing about this document was special-cased.
hit = None
for attempt in range(6):
    res = documind_tools.retrieve("What is the rehearsal reference code for this ingestion run?", TENANT, top_k=5)
    hit = next((c for c in res.get("citations", []) if FACT_CODE in (c.get("quote") or c.get("text") or "")), None)
    if hit or FACT_CODE in (res.get("answer") or ""):
        break
    time.sleep(10)
print("answerable:", res.get("answerable"), "| answer:", (res.get("answer") or "")[:120])
print("cited     :", [(c.get("source") or c.get("source_uri", "")).split("/")[-1] for c in res.get("citations", [])][:3])
assert FACT_CODE in (res.get("answer") or "") or hit, "the fact uploaded three cells ago must come back through the door"


## Cell 4: The same bytes, again
> **The idempotency key is a hash of the content, never the Pub/Sub message id.** The message id is stable across redelivery, so it would deduplicate a retry - but the same file uploaded twice is two messages with two ids, and that is the duplicate users actually create.


In [ ]:
# THE SAME BYTES, AGAIN. A second upload is a second Pub/Sub message with a second id - the message id would not
# catch it - but the same content hash, and claim() refuses it: the worker returns 200 {"status": "duplicate"}, which
# ACKS the message (a success, not a failure). Nothing is re-parsed, re-embedded or re-billed, and the chunk ids being
# deterministic (tenant:sha256#i) means even a forced re-run would UPSERT, never add a second copy.
before = len(chunks_of(uri))
since = datetime.datetime.now(datetime.timezone.utc)
uri2 = upload(f"module12_rehearsal_note_{STAMP}_again.md", NOTE, "text/markdown")
print("uploaded the same bytes as", uri2.split("/")[-1])
time.sleep(45)
again = wait_for("ingest_ok", since, minutes=1)
after = len(chunks_of(uri))
claim = db.collection("documents").document(doc.doc_key).get().to_dict()
print(f"chunks under the first object: {before} -> {after} | a second ingest_ok: {bool(again)} | claim status: {claim.get('status')} gcs_uri: {claim.get('gcs_uri', '').split('/')[-1]}")
assert not again and after == before, "the duplicate must be acked without indexing"
print("the claim still names the FIRST object: the second upload was answered from the claim, not the index")


### The claim, as a race
The transaction in `idempotency.py` is what makes the cell above deterministic. Against a fake Firestore, so the sequence is visible without the plumbing:


In [ ]:
# claim() against a fake Firestore, so the RACE is visible without a project.
class FakeDoc:
    def __init__(self, store, key): self.store, self.key = store, key
    @property
    def exists(self): return self.key in self.store

class FakeRef:
    def __init__(self, store, key): self.store, self.key = store, key
    def get(self, transaction=None): return FakeDoc(self.store, self.key)
    def set(self, data, merge=False):
        self.store.setdefault(self.key, {}).update(data) if merge else \
            self.store.__setitem__(self.key, dict(data))

class FakeDB:
    def __init__(self): self.store = {}
    def collection(self, _): return self
    def document(self, key): return FakeRef(self.store, key)
    def transaction(self): return object()

# The real claim() uses @firestore.transactional; here we call the same logic
# directly so the SEQUENCE is what you can see, not the plumbing.
def claim(db, doc_key, gcs_uri):
    ref = db.collection("documents").document(doc_key)
    if ref.get().exists:
        return False
    ref.set({"gcs_uri": gcs_uri, "status": "processing"})
    return True

def release(db, doc_key, error):
    db.collection("documents").document(doc_key).set(
        {"status": "failed", "error": error}, merge=True)

db = FakeDB()
KEY = "acme_3f2b9c...";  URI = "gs://documind-uploads/acme/hr_policy_2026.pdf"

print("delivery 1 (first ever):     ", claim(db, KEY, URI))   # True  - process it
print("delivery 2 (Pub/Sub retry):  ", claim(db, KEY, URI))   # False - ack, do nothing
print("re-upload of the same bytes: ", claim(db, KEY, URI))   # False - same key

# Now the failure that used to strand documents for ever: the worker crashed
# between claim and finish, so the claim is still held.
db.store[KEY]["status"] = "processing"      # as it would be after a crash
print("\nafter a crash, retry sees:  ", claim(db, KEY, URI), "<- stuck for ever")
release(db, KEY, "TimeoutError")
del db.store[KEY]                            # release + a reaper clears the claim
print("after release + reaper:      ", claim(db, KEY, URI), "<- retry can proceed")


## Cell 5: Poison
> **400, not 500.** 500 tells Pub/Sub "try again"; a message this worker can never parse fails identically five times, burns the backoff window, and only then reaches the DLQ. 400 acks it at once, and the dead-letter policy still captures it by delivery count.


In [ ]:
# POISON. A zero-byte object is a message the worker can parse - IngestMessage refuses size=0 at the edge - and the
# worker answers 400, not 500: a message that will fail identically every time must not be retried five times with
# backoff before the DLQ sees it. The ingest_poison line is the proof; the dead-letter subscription is where the
# message lands after Pub/Sub's own delivery attempts (eventarc.tf: five, 10 s to 600 s), minutes later.
since = datetime.datetime.now(datetime.timezone.utc)
bad = upload(f"poison_{STAMP}.pdf", b"", "application/pdf")
print("uploaded", bad.split("/")[-1], "(zero bytes)")
line = wait_for("ingest_poison", since, tenant=None, minutes=2)
assert line, "no ingest_poison line in 2 min"
print("the worker refused it (400):", line.get("error"))
print()
print("the dead-letter subscription, peeked without acking (make dlq):")
print(gcloud("pubsub", "subscriptions", "pull", "ingest-dlq-sub", "--limit", "5",
             "--format=table(message.messageId,message.attributes.objectId,message.attributes.eventTime,deliveryAttempt)")
      or "(empty: Pub/Sub is still retrying - eventarc.tf's backoff runs 10 s to 600 s; re-run this cell in a few minutes)")


### The poison path, in shape


In [ ]:
# The poison path, and why it returns 400 rather than 500.
import base64, json

def push(envelope):
    """The worker's entry point, minus GCS and Doc AI."""
    try:
        raw = base64.b64decode(envelope["message"]["data"])
        msg = IngestMessage.model_validate_json(raw)
    except Exception as e:
        return 400, f"unparseable: {type(e).__name__}"
    return 200, f"accepted {msg.gcs_uri}"

good = IngestMessage(bucket=UPLOAD_BUCKET, name=f"{TENANT}/hr_policy_2026.pdf", size=48_000, content_type="application/pdf", generation="17", tenant_id=TENANT)
ok = {"message": {"data": base64.b64encode(good.model_dump_json().encode()).decode()}}
poison = {"message": {"data": base64.b64encode(b'{"bucket": "u"}').decode()}}
notjson = {"message": {"data": base64.b64encode(b'<html>404</html>').decode()}}

for name, env in (("valid", ok), ("missing fields", poison), ("not json", notjson)):
    print(f"  {name:16} -> HTTP {push(env)[0]}  {push(env)[1][:52]}")

# WHY 400 AND NOT 500:
#   500 tells Pub/Sub "try again". A message this worker can never parse will
#   fail identically five times, burn the backoff window, and only THEN reach
#   the DLQ - twenty minutes of retrying something that cannot succeed.
#   400 acks it immediately; the dead_letter_policy still captures it because
#   the message is nacked into the DLQ by delivery count, and the operator sees
#   it in ingest-dlq rather than in a latency graph.
#
# The distinction is: 500 means "this might work later", 400 means "this will
# never work". Retrying the second kind is how a poison message takes a queue
# down.


## Cell 6: A figure is a document


In [ ]:
import io
from PIL import Image, ImageDraw

# A FIGURE IS A DOCUMENT (Module 9). A PNG under the same prefix takes the media path: no Document AI, no chunker -
# Gemini describes the picture, the description becomes ONE chunk of kind figure with media_url set, and the pixels
# are scanned by DLP in the image region before anything is indexed. A citation of it renders inline in the UI (12.4).
img = Image.new("RGB", (640, 320), "white")
d = ImageDraw.Draw(img)
d.rectangle([40, 60, 600, 260], outline="black", width=3)
for i, (label, h) in enumerate((("Year 1", 40), ("Year 3", 90), ("Year 5", 150), ("Year 7", 190))):
    x = 90 + i * 130
    d.rectangle([x, 250 - h, x + 80, 250], fill="#0d9488")
    d.text((x, 255), label, fill="black")
d.text((40, 20), f"Figure: gratuity accrual by years of service (rehearsal {STAMP})", fill="black")
buf = io.BytesIO(); img.save(buf, format="PNG")
since = datetime.datetime.now(datetime.timezone.utc)
png = upload(f"module12_figure_{STAMP}.png", buf.getvalue(), "image/png")
line = wait_for("ingest_ok", since)
assert line, "no ingest_ok for the PNG in 5 min"
print("the worker's line:", {k: line.get(k) for k in ("doc_key", "chunks", "pages", "kinds")})
rows = chunks_of(png)
c = rows[0]
print("the chunk:", {"kind": c.get("kind"), "doc_type": c.get("doc_type"), "media_url": (c.get("media_url") or "")[:60], "caption": c["text"][:120]})
assert c.get("kind") == "figure" and c.get("media_url"), "a PNG becomes one figure chunk with a locator"


## Cell 7: The contracts, exercised
Against the module the lane runs - imported from the clone - so if `contracts.py` is broken, this is where you find out.


In [ ]:
from pydantic import ValidationError

# THE CONTRACTS, EXERCISED - against the module the LANE runs, imported from the clone, not a copy pasted here.
good = IngestMessage(bucket=UPLOAD_BUCKET, name=f"{TENANT}/hr_policy_2026.pdf", size=48_000, content_type="application/pdf",
                     generation="17", tenant_id=TENANT)
print("parsed:", good.gcs_uri)
pdf = b"%PDF-1.7 ... notice period 60 days ..."
a = DocumentContract(tenant_id=TENANT, sha256=sha256_of(pdf), gcs_uri=f"gs://{UPLOAD_BUCKET}/{TENANT}/first.pdf", pages=48)
b = DocumentContract(tenant_id=TENANT, sha256=sha256_of(pdf), gcs_uri=f"gs://{UPLOAD_BUCKET}/{TENANT}/second-upload.pdf", pages=48)
print("same bytes, different object names -> same doc_key:", a.doc_key == b.doc_key, "|", a.doc_key[:34], "...")
print("chunk ids are deterministic and tenant-scoped:", a.chunk_id(0) == b.chunk_id(0), a.chunk_id(0)[:24])
for bad in ({"bucket": "u", "name": "../../etc/passwd", "size": 1, "content_type": "application/pdf", "generation": "1", "tenant_id": TENANT},
            {"bucket": "u", "name": f"{TENANT}/x.pdf", "size": 0, "content_type": "application/pdf", "generation": "1", "tenant_id": TENANT}):
    try:
        IngestMessage(**bad)
        print("ACCEPTED - the validator is not doing its job")
    except ValidationError as e:
        print("rejected:", e.errors()[0]["loc"][0], "-", e.errors()[0]["msg"][:48])


## Cell 8: A policy changes
> **The index has a current.** A document has an identity (its object path) and versions (its content hashes). Re-issue it under the same name and the worker indexes the new version, retires the old one's chunks (a flag, never a delete), records the source in `sources/`, and drops the tenant's cache record. Upload the old bytes again and they come back without a re-embedding. The date a document declares rides on its chunks, into the context and the stream.


In [ ]:
# A POLICY CHANGES (the ledger, 11 September 2026). The handbook is re-issued under its OWN object name: NP-03's notice
# period becomes 90 days, dated 1 October 2026. The worker indexes revision 2 as current, RETIRES revision 1's chunks -
# flagged, never deleted - records the source in sources/, drops the tenant's cache record, and says so in one line.
# Then the undo: the original bytes again, and revision 1 comes back without a single re-embedding. The lane ends
# where it began, so the golden row that expects 60 days (lk-06) stays green for the next lesson.
V2 = open(f"{KIT}/deploy/evals/demo/hr_policy_2026_v2.md", "rb").read()
V1 = open(f"{KIT}/deploy/evals/corpus/acme/hr_policy_2026.md", "rb").read()
NAME = "hr_policy_2026.md"                                     # the SAME object name: a document re-issued, not a new one
Q = "What is the notice period for a confirmed E3?"
ledger = db.collection("sources").document(f"{TENANT}~{NAME}")
uri = f"gs://{UPLOAD_BUCKET}/{TENANT}/{NAME}"

def versions() -> dict:
    """current / retired chunk counts for the object, by version - the ledger's promise, read from the index."""
    out = {}
    for d in chunks_of(uri):
        key = d.get("doc_key") or "pre-ledger"
        out.setdefault(key, {"current": 0, "retired": 0})["retired" if d.get("current") is False else "current"] += 1
    return out

def ask() -> str:
    st, ans = api("/v1/query", {"query": Q, "tenant_id": TENANT, "user_id": "u_12_5", "top_k": 5, "stream": False})
    return (ans.get("answer", "") if isinstance(ans, dict) else str(ans))[:160].replace(chr(10), " ")

print("before     :", versions(), "| ledger row:", (ledger.get().to_dict() or {}).get("doc_key", "none yet (a lane older than the ledger: make backfill-current)"))
since = datetime.datetime.now(datetime.timezone.utc)
upload(NAME, V2, "text/markdown")
line = wait_for("ingest_ok", since)
assert line, "no ingest_ok in 5 min"
print("the line   :", {k: line.get(k) for k in ("doc_key", "chunks", "retired", "effective_from")})
if not line.get("retired"):
    print("the worker on the lane predates the ledger (no retired count). Cloud Shell: make build deploy-services SERVICES_lean=ingest"
          " SCRIPTS_lean=commands/lesson-12.5.sh, make backfill-current, then re-run. Restoring revision 1 now.")
    upload(NAME, V1, "text/markdown")
else:
    row = ledger.get().to_dict()
    print("the ledger :", {k: row.get(k) for k in ("doc_key", "generation", "effective_from", "chunks", "status")})
    print("after      :", versions())
    assert row["doc_key"] == line["doc_key"] and line["effective_from"] == "2026-10-01"
    assert all(v["current"] == 0 for k, v in versions().items() if k != line["doc_key"]), "only revision 2 is current"
    time.sleep(10)
    a2 = ask()
    print("revision 2 :", a2)
    if "90" not in a2:
        print("  (both versions were retrieved: the API on the lane predates the ledger - make build deploy-services SERVICES_lean=api)")
    # THE UNDO: the original bytes again. Their chunks are still here, flagged; the worker flips them back
    # (ingest_reactivated) and retires revision 2 - nothing is re-embedded, because nothing was ever deleted.
    since = datetime.datetime.now(datetime.timezone.utc)
    upload(NAME, V1, "text/markdown")
    back = wait_for("ingest_reactivated", since, minutes=2) or wait_for("ingest_ok", since, minutes=3)
    assert back, "no ingest line for the undo in 5 min"
    print("the undo   :", back.get("event"), {k: back.get(k) for k in ("doc_key", "chunks", "retired")}, "| now:", versions())
    time.sleep(10)
    a1 = ask()
    print("revision 1 :", a1)
    assert "60" in a1, "the lane ends where it began: lk-06 stays green"
print("\nmake reindex FILE=evals/demo/hr_policy_2026_v2.md NAME=hr_policy_2026.md TENANT=acme   # the same from Cloud Shell; make reconcile is the nightly half")


### Reconciliation, the other half
The worker cannot see a deletion or a lost event; a nightly walk of the bucket against the ledger can. The planner is pure, so it runs here on a made-up bucket before the selftest runs from the clone.


In [ ]:
# RECONCILIATION - THE WALK THE WORKER CANNOT DO. The worker keeps the ledger on the way in, one object at a time; what
# it never sees is an object deleted from the bucket, or one overwritten while its event was lost. reconcile.py is the
# other half every production indexer has (LangChain's full cleanup, Vertex AI Search's FULL reconciliation, Bedrock's
# sync): the bucket's current generations against sources/. The planner is pure - it runs here on a made-up bucket -
# then the module's own selftest runs from the clone, and reconcile.tf's nightly job is read back: 23:30 IST, after
# documind-off has floored the lane, as the worker's own account, and only when RECONCILE_JOB=true.
import reconcile

bucket = [{"name": f"{TENANT}/hr_policy_2026.md", "generation": "9", "tenant_id": TENANT},   # in the ledger, unchanged
          {"name": f"{TENANT}/msa_zeta_2026.md", "generation": "12", "tenant_id": TENANT},  # a newer generation than the ledger saw
          {"name": f"{TENANT}/new_handbook.pdf", "generation": "1", "tenant_id": TENANT}]   # never in the ledger
ledger = {f"{TENANT}~hr_policy_2026.md": {"gcs_uri": f"gs://{UPLOAD_BUCKET}/{TENANT}/hr_policy_2026.md", "doc_key": f"{TENANT}_s1", "generation": "9", "sha256": "s1", "status": "indexed"},
          f"{TENANT}~msa_zeta_2026.md": {"gcs_uri": f"gs://{UPLOAD_BUCKET}/{TENANT}/msa_zeta_2026.md", "doc_key": f"{TENANT}_s2", "generation": "11", "sha256": "s2", "status": "indexed"},
          f"{TENANT}~old_circular.pdf": {"gcs_uri": f"gs://{UPLOAD_BUCKET}/{TENANT}/old_circular.pdf", "doc_key": f"{TENANT}_s3", "generation": "3", "sha256": "s3", "status": "indexed",
                                         "name": f"{TENANT}/old_circular.pdf", "tenant_id": TENANT}}   # gone from the bucket
documents = {f"{TENANT}_s9": {"status": "indexed", "gcs_uri": f"gs://{UPLOAD_BUCKET}/{TENANT}/new_handbook.pdf"}}   # indexed before the ledger existed
for a in reconcile.plan(bucket, ledger, documents):
    print(f"  {a['action']:12} {a['name']:36} {a.get('why', '')}")
print()
# check_bytes is decided once the object is hashed: the same sha under a new generation is a touch (record it, index
# nothing); a sha the per-version claims already know is a backfill (the ledger never heard of it); anything else is
# a re-ingest - the object rewritten onto itself, so the worker's ONE path runs, not a second one.
for sha, row in (("s2", ledger[f"{TENANT}~msa_zeta_2026.md"]), ("s9", None), ("s8", ledger[f"{TENANT}~msa_zeta_2026.md"])):
    print(f"  bytes {sha}: {reconcile.decide_bytes(sha, TENANT, row, documents)}")
print()
r = subprocess.run([sys.executable, f"{KIT}/deploy/services/ingest/reconcile.py", "--selftest"], capture_output=True, text=True)
assert r.returncode == 0, r.stdout + r.stderr
print(r.stdout.strip())
print()
print(excerpt("terraform/reconcile.tf", "google_cloud_scheduler_job", after=8))
print()
print("make reconcile TENANT_ONLY=acme prints this plan against the real bucket; APPLY=1 acts; make reconcile-job builds the")
print("job from the ingest image, and RECONCILE_JOB=true on the next apply is what schedules it - the job, then the schedule")


## Where this goes
- **12.6**'s DLP list is the one the worker scanned every chunk and the PNG's pixels with; **12.3** listed it.
- **12.7** ships a new worker image the same way it ships the API: a candidate revision, a gate, a traffic flip.
- **12.2**'s `RETRIEVAL_CURRENT_ONLY=on` is the switch that makes the ledger's promise a pre-filter; `make reconcile` is its nightly half.

## ✅ Lesson 12.5 complete
- ✅ A new document ingested end to end: the worker's ingest_ok line, the claim in Firestore, the fact retrieved through the one retrieve()
- ✅ The same bytes refused by the claim: no second line, the chunk count unchanged
- ✅ A zero-byte object refused at the edge with a 400; the dead-letter subscription peeked
- ✅ A PNG turned into one figure chunk with a locator
- ✅ The contracts, the claim race and the poison path, against the lane's own modules
- ✅ A document re-issued under its own name: revision 2 current, revision 1 retired, the ledger row, the answer moved, the undo without a re-embedding
- ✅ The reconciliation planned offline (retire, re-ingest, backfill, touch), the selftest run from the clone, the nightly job read back from reconcile.tf


## The files this lesson owns
Below are the twelve heredocs the extractor turns into `deploy/services/ingest/*`, the ingestion Terraform and the worker's deploy: the contracts, the claim and the ledger, the parser, the indexer, the worker, Vector Search (full profile), the Document AI processor, the Firestore indexes (two vector indexes on `chunks` now: the second carries `current`), the Eventarc lane with its floor, the requirements, the Dockerfile and the deploy. The ledger's edits of 11 September reached them through `readopt`; the story above uploaded four objects into the lane they describe.


In [ ]:
CONTRACTS_PY = '''
"""Contracts for the ingest lane. Everything crossing a queue is validated."""
import hashlib
import re

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator


class IngestMessage(BaseModel):
    """The object record Cloud Storage publishes on object.finalized (eventarc.tf's
    notification, payload JSON_API_V1), as we choose to see it.

    Pub/Sub hands you whatever the publisher sent. Parsing it into a model at
    the edge means a malformed message fails HERE, loudly, with a field name -
    instead of three functions later with a KeyError nobody can place.

    The record spells the type `contentType` and its size as a string of digits;
    the alias and pydantic's coercion take both. It carries no tenant: the object
    PATH does - `acme/code_on_wages_2019.pdf` - and the first segment is the
    tenant, which is why evals/upload.sh puts one prefix per tenant and why an
    object at the bucket root is poison, not a default tenant.
    """
    model_config = ConfigDict(populate_by_name=True)

    bucket: str
    name: str
    size: int = Field(ge=1)
    content_type: str = Field(alias="contentType")
    generation: str
    tenant_id: str = ""

    @field_validator("name")
    @classmethod
    def _no_traversal(cls, v: str) -> str:
        if ".." in v or v.startswith("/"):
            raise ValueError("object name escapes its prefix")
        return v

    @model_validator(mode="after")
    def _tenant_from_path(self):
        if not self.tenant_id:
            prefix, sep, rest = self.name.partition("/")
            if not sep or not prefix or not rest:
                raise ValueError("object is not under a tenant prefix")
            self.tenant_id = prefix
        return self

    @property
    def gcs_uri(self) -> str:
        return f"gs://{self.bucket}/{self.name}"


class DocumentContract(BaseModel):
    """What one ingested document looks like once it exists.

    doc_key is the IDEMPOTENCY KEY and it is derived from CONTENT, never from
    the Pub/Sub message id. The message id is stable across REDELIVERY, so it
    would deduplicate a retry - but the same PDF uploaded twice is two
    different messages with two different ids, and that is the duplicate
    users actually create. A content hash catches both.
    """
    tenant_id: str
    sha256: str
    gcs_uri: str
    pages: int
    doc_type: str = "unknown"
    # The ledger (11 September 2026): when the document says it applies from. Declared by the document, never
    # guessed by the pipeline; absent means undated. Carried onto every chunk and every citation.
    effective_from: str | None = None

    @property
    def doc_key(self) -> str:
        return f"{self.tenant_id}_{self.sha256}"

    def chunk_id(self, i: int) -> str:
        # Deterministic too: a re-run UPSERTS the same ids instead of adding a
        # second copy of every chunk beside the first.
        #
        # And TENANT-SCOPED, like doc_key. Until 7 Sept 2026 this was f"{sha256}#{i}": two
        # tenants uploading the same Act (the corpus shares seven documents between ACME and
        # Zeta or Globex) wrote the same Firestore documents, and the second ingest overwrote
        # the first tenant's chunks with its own tenant_id. ACME lost every shared document
        # to whichever tenant ingested it last, and the live eval refused eight questions
        # about documents ACME had uploaded. The colon matches the ids the notebooks mint
        # (acme:hr_policy_2026#NP-03); the same id in one tenant's index and another's is a
        # collision, never a saving.
        return f"{self.tenant_id}:{self.sha256}#{i}"


def sha256_of(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


# A document declares its own effective date: `effective_from: 2026-10-01` (or `Effective from: ...`) in its first
# lines - Markdown front matter or a header line - or `_effective_2026-10-01` in its object name for a PDF nobody
# can edit. Two documents that disagree are then a question of dates, not of which one the reranker liked.
_EFFECTIVE_TEXT = re.compile(r"effective[ _-]?(?:from|date)?\\s*[:=]\\s*(\\d{4}-\\d{2}-\\d{2})", re.I)
_EFFECTIVE_NAME = re.compile(r"_effective_(\\d{4}-\\d{2}-\\d{2})\\.")


def effective_from_of(name: str, text: str | None) -> str | None:
    """The date a document declares, from its object name first, then its first lines; None when undated."""
    m = _EFFECTIVE_NAME.search(name or "")
    if m:
        return m.group(1)
    if text:
        m = _EFFECTIVE_TEXT.search(text[:3000])
        if m:
            return m.group(1)
    return None
'''

with open('contracts.py', 'w') as f: f.write(CONTRACTS_PY)
print('wrote contracts.py')


In [ ]:
IDEMPOTENCY_PY = '''
"""The claim: exactly one worker may process a given document - and the ledger: one current version per document."""
import logging

from google.cloud import firestore

log = logging.getLogger("documind.ingest")


def claim(db: firestore.Client, doc_key: str, gcs_uri: str) -> bool:
    """Claim doc_key. True if THIS caller may proceed, False if someone already did.

    The transaction is the whole point. Read-then-write without one is a race
    with a window measured in milliseconds - and Pub/Sub delivers duplicates
    concurrently, so it is a window that gets hit. Two workers both read
    "absent", both write, and the document is ingested twice.
    """
    ref = db.collection("documents").document(doc_key)

    @firestore.transactional
    def _claim(tx: firestore.Transaction) -> bool:
        snap = ref.get(transaction=tx)
        # A claim that ended in `failed` is not a claim, it is a record of one. release()
        # writes it so the error is readable; the NEXT delivery must be allowed to try
        # again, or "give the claim back" gave nothing back: on the first live load every
        # retry of a failed document was acked as a duplicate and the document stayed failed.
        if snap.exists and snap.get("status") != "failed":
            return False
        tx.set(ref, {"gcs_uri": gcs_uri,
                     "status": "processing",
                     "claimed_at": firestore.SERVER_TIMESTAMP})
        return True

    won = _claim(db.transaction())
    if not won:
        log.info('{"event":"ingest_duplicate","doc_key":"%s"}', doc_key)
    return won


def finish(db: firestore.Client, doc_key: str, chunks: int) -> None:
    db.collection("documents").document(doc_key).set(
        {"status": "indexed", "chunks": chunks,
         "indexed_at": firestore.SERVER_TIMESTAMP}, merge=True)


def release(db: firestore.Client, doc_key: str, error: str) -> None:
    """Give the claim back so a retry can take it.

    Without this a crash between claim and finish leaves the document claimed
    for ever, and every redelivery sees "already processing" and does nothing.
    The document is then permanently missing and nothing is alerting, because
    from Pub/Sub's point of view every delivery was acked successfully.
    """
    db.collection("documents").document(doc_key).set(
        {"status": "failed", "error": error[:400],
         "failed_at": firestore.SERVER_TIMESTAMP}, merge=True)


# ------------------------------------------------------------------------------ the ledger (11 September 2026)
# A document has an IDENTITY - its object path - and VERSIONS - the content hashes. `documents/{doc_key}` above is
# the per-version claim; `sources/{source_id}` is the per-document ledger: which version is current, its generation,
# when it was indexed, the date it declares. It is what every production indexer keeps (a record manager keyed on the
# source), and it is what lets a re-issued document RETIRE its predecessor's chunks instead of standing beside them.
# Retiring is a flag, never a delete: the audit story survives, the full profile's mirrors keep their history, and a
# bad re-index is undone by uploading the previous bytes again (reactivate) - nothing is re-embedded.


def source_id_for(tenant_id: str, name: str) -> str:
    """Firestore ids cannot hold '/'; the object path already starts with the tenant prefix."""
    return name.replace("/", "~")


def _doc_key_of(snap) -> str:
    d = snap.to_dict() or {}
    # Chunks written before the ledger carry no doc_key; their id is <tenant>:<sha256>#<i>.
    return d.get("doc_key") or snap.id.split("#")[0].replace(":", "_", 1)


def retire_previous(db: firestore.Client, tenant_id: str, gcs_uri: str, keep_doc_key: str | None,
                    chunks_collection: str = "chunks") -> dict:
    """Retire every chunk of `gcs_uri` that is not `keep_doc_key`: current=false, superseded_by, superseded_at.

    Runs AFTER the new version's chunks are written, so a reader never sees a document with no current version.
    keep_doc_key=None retires the whole source (reconcile: the object is gone from the bucket). Two equality filters
    need no composite index. Returns the retired keys, ids and count - the worker removes the ids from Vector Search
    on the full profile and logs the count."""
    retired_keys, retired_ids = set(), []
    batch, pending = db.batch(), 0
    query = (db.collection(chunks_collection).where("tenant_id", "==", tenant_id)
             .where("source_uri", "==", gcs_uri))
    for snap in query.stream():
        key = _doc_key_of(snap)
        if (keep_doc_key and key == keep_doc_key) or (snap.to_dict() or {}).get("current") is False:
            continue
        batch.update(snap.reference, {"current": False, "superseded_by": keep_doc_key,
                                      "superseded_at": firestore.SERVER_TIMESTAMP})
        retired_keys.add(key)
        retired_ids.append(snap.id)
        pending += 1
        if pending == 400:
            batch.commit()
            batch, pending = db.batch(), 0
    if pending:
        batch.commit()
    for key in retired_keys:
        db.collection("documents").document(key).set(
            {"status": "superseded", "superseded_by": keep_doc_key,
             "superseded_at": firestore.SERVER_TIMESTAMP}, merge=True)
    return {"retired_doc_keys": sorted(retired_keys), "retired_ids": retired_ids,
            "retired_chunks": len(retired_ids)}


def reactivate(db: firestore.Client, tenant_id: str, gcs_uri: str, doc_key: str,
               chunks_collection: str = "chunks") -> int:
    """The undo. The same bytes uploaded again after a newer version retired them: their chunks are still here,
    flagged, so flipping the flag back is a re-index that costs nothing. The caller retires the newer version next."""
    n, batch, pending = 0, db.batch(), 0
    query = (db.collection(chunks_collection).where("tenant_id", "==", tenant_id)
             .where("source_uri", "==", gcs_uri))
    for snap in query.stream():
        if _doc_key_of(snap) == doc_key and (snap.to_dict() or {}).get("current") is False:
            batch.update(snap.reference, {"current": True, "superseded_by": firestore.DELETE_FIELD,
                                          "superseded_at": firestore.DELETE_FIELD,
                                          "reactivated_at": firestore.SERVER_TIMESTAMP})
            n += 1
            pending += 1
            if pending == 400:
                batch.commit()
                batch, pending = db.batch(), 0
    if pending:
        batch.commit()
    db.collection("documents").document(doc_key).set(
        {"status": "indexed", "superseded_by": firestore.DELETE_FIELD,
         "reactivated_at": firestore.SERVER_TIMESTAMP}, merge=True)
    return n


def status_of(db: firestore.Client, doc_key: str) -> str | None:
    snap = db.collection("documents").document(doc_key).get()
    return (snap.to_dict() or {}).get("status") if snap.exists else None


def record_source(db: firestore.Client, tenant_id: str, name: str, gcs_uri: str, doc_key: str,
                  generation: str, sha256: str, chunks: int, effective_from: str | None = None,
                  status: str = "indexed") -> None:
    """The ledger row: what is current for this object path, and since when."""
    db.collection("sources").document(source_id_for(tenant_id, name)).set(
        {"tenant_id": tenant_id, "name": name, "gcs_uri": gcs_uri, "doc_key": doc_key,
         "generation": str(generation), "sha256": sha256, "chunks": chunks,
         "effective_from": effective_from, "status": status,
         "indexed_at": firestore.SERVER_TIMESTAMP}, merge=True)


def drop_tenant_cache(db: firestore.Client, tenant_id: str) -> bool:
    """The cache follows the index. The worker cannot delete the Vertex cache object (that is the API's
    permission) - it deletes the RECORD the API reads, so the next answer runs uncached and `make cache` rebuilds
    the pack from the corpus that changed. Returns whether there was a record to drop."""
    ref = db.collection("tenant_caches").document(tenant_id)
    if not ref.get().exists:
        return False
    ref.delete()
    log.info('{"event":"cache_record_dropped","tenant":"%s"}', tenant_id)
    return True
'''

with open('idempotency.py', 'w') as f: f.write(IDEMPOTENCY_PY)
print('wrote idempotency.py')


In [ ]:
PARSER_PY = '''
"""Doc AI, chosen by residency rather than by preference."""
import io
import os

from google.cloud import documentai
from pypdf import PdfReader, PdfWriter

RESIDENCY = os.environ.get("RESIDENCY", "india")     # india | us

# Layout Parser gives you document STRUCTURE - headings, tables, reading order -
# and it runs in `us` only. Enterprise OCR runs in asia-south1 and gives you
# text plus layout boxes, no semantic structure.
#
# For a DPDP deployment that is not a trade-off you get to make on quality
# grounds: if the document carries personal data of people in India and the
# customer's contract says it stays in India, the processor that runs in `us`
# is not available to you, whatever it would have given you.
PROCESSORS = {
    "india": {"location": "asia-south1", "type": "OCR_PROCESSOR"},
    "us":    {"location": "us",          "type": "LAYOUT_PARSER_PROCESSOR"},
}

# An online (synchronous) request takes at most 15 pages, for OCR, Layout Parser and Form
# Parser alike (docs.cloud.google.com/document-ai/limits). The kit's corpus is thirteen
# Acts averaging 53 pages, so a PDF goes up in 15-page slices - lesson 4.1's slices(), the
# same limit - and its pages come back in order. Batch processing takes 500 pages but is
# asynchronous and needs an output prefix to poll; a dozen online calls inside one push
# request is simpler and stays inside the 600-second ack deadline eventarc.tf sets.
ONLINE_PAGE_LIMIT = 15


def processor_config() -> dict:
    return PROCESSORS[RESIDENCY]


def _client_and_name(project_id: str, processor_id: str):
    cfg = processor_config()
    client = documentai.DocumentProcessorServiceClient(
        client_options={"api_endpoint": f"{cfg['location']}-documentai.googleapis.com"})
    return client, f"projects/{project_id}/locations/{cfg['location']}/processors/{processor_id}"


def _page_texts(doc) -> list[str]:
    """One string per page, so the worker can put a form feed between pages and _chunk()
    can name the page a chunk starts on - the page_start a citation shows.

    OCR fills `pages`, each with a text anchor into `text`. Layout Parser fills
    `document_layout` instead: blocks with their own text and a page span. Either way the
    result is pages in order; a document with neither is one page of whatever text it has."""
    if doc.pages:
        pages = []
        for page in doc.pages:
            parts = [doc.text[int(seg.start_index):int(seg.end_index)]
                     for seg in page.layout.text_anchor.text_segments]
            pages.append("".join(parts))
        return pages
    by_page: dict[int, list[str]] = {}

    def walk(blocks):
        for block in blocks:
            page = int(block.page_span.page_start or 1)
            if block.text_block.text:
                by_page.setdefault(page, []).append(block.text_block.text)
            walk(block.text_block.blocks)
            for row in list(block.table_block.header_rows) + list(block.table_block.body_rows):
                for cell in row.cells:
                    walk(cell.blocks)
            if block.list_block.list_entries:
                for entry in block.list_block.list_entries:
                    walk(entry.blocks)

    walk(doc.document_layout.blocks)
    if by_page:
        last = max(by_page)
        return ["\\n".join(by_page.get(p, [])) for p in range(1, last + 1)]
    return [doc.text]


def _process(client, name: str, content: bytes, mime_type: str) -> list[str]:
    result = client.process_document(
        request=documentai.ProcessRequest(
            name=name,
            raw_document=documentai.RawDocument(content=content, mime_type=mime_type)))
    return _page_texts(result.document)


def parse(project_id: str, processor_id: str, content: bytes,
          mime_type: str) -> tuple[str, int]:
    """Return (text with a form feed between pages, page_count)."""
    client, name = _client_and_name(project_id, processor_id)
    if mime_type == "application/pdf":
        reader = PdfReader(io.BytesIO(content))
        if len(reader.pages) > ONLINE_PAGE_LIMIT:
            pages: list[str] = []
            for first in range(0, len(reader.pages), ONLINE_PAGE_LIMIT):
                writer = PdfWriter()
                for page in reader.pages[first:first + ONLINE_PAGE_LIMIT]:
                    writer.add_page(page)
                buf = io.BytesIO()
                writer.write(buf)
                pages.extend(_process(client, name, buf.getvalue(), mime_type))
            return "\\f".join(pages), len(pages)
    pages = _process(client, name, content, mime_type)
    return "\\f".join(pages), len(pages)
'''

with open('parser.py', 'w') as f: f.write(PARSER_PY)
print('wrote parser.py')


In [ ]:
INDEXER_PY = '''
"""Embed, upsert to Vector Search, mirror into Firestore."""
import os

from google.cloud import aiplatform
from google.cloud import firestore
from google.cloud.aiplatform_v1.types import IndexDatapoint
from google.cloud.firestore_v1.vector import Vector
from google import genai

EMBED_BATCH = 250          # the regional API's per-request ceiling, in texts
# ... and in tokens: text-embedding-005 takes at most 20,000 tokens per REQUEST, across all
# the texts in it. Forty 500-token chunks is 20,000; every long Act failed on exactly this
# the first time the corpus was loaded. Tokens are estimated at three characters each -
# an overestimate for English, so a batch stops early rather than late.
EMBED_TOKENS = 15_000
CHARS_PER_TOKEN = 3
DRY_RUN = os.environ.get("VECTOR_DRY_RUN") == "1"

# Embeddings are REGIONAL. Generation is global-only; this client is neither
# interchangeable with that one nor optional to get right.
_embed = genai.Client(enterprise=True,
                      project=os.environ["GOOGLE_CLOUD_PROJECT"],
                      location="us-central1")


def batches(texts: list[str]) -> list[list[str]]:
    """Batches of at most EMBED_BATCH texts and about EMBED_TOKENS tokens. Send 251 texts, or
    20,001 tokens, and the request fails - not the last item, the whole call - so a
    300-chunk document would index nothing at all."""
    out, cur, cur_tokens = [], [], 0
    for t in texts:
        tokens = max(1, len(t) // CHARS_PER_TOKEN)
        if cur and (len(cur) >= EMBED_BATCH or cur_tokens + tokens > EMBED_TOKENS):
            out.append(cur); cur, cur_tokens = [], 0
        cur.append(t); cur_tokens += tokens
    if cur:
        out.append(cur)
    return out


def embed_all(texts: list[str]) -> list[list[float]]:
    out: list[list[float]] = []
    for batch in batches(texts):
        r = _embed.models.embed_content(
            model="text-embedding-005", contents=batch,
            config={"output_dimensionality": 768})
        out.extend([e.values for e in r.embeddings])
    return out


def to_datapoints(doc, chunks: list[dict],
                  vectors: list[list[float]]) -> list[IndexDatapoint]:
    """chunks are dicts - {text, kind, media_url?, page_start?, start?, end?} - since the
    corpus grew figures and video segments (9.6). Only the text is embedded."""
    return [
        IndexDatapoint(
            datapoint_id=doc.chunk_id(i),
            feature_vector=v,
            # The tenant restrict is what makes one index safe for many
            # customers. It is set HERE, at write time - a filter applied only
            # at query time is one forgotten WHERE clause away from a leak.
            # `kind` is a second restrict so a caller can ask for figures only
            # (filters={"kind": "figure"} in rag-api's QueryRequest).
            restricts=[IndexDatapoint.Restriction(
                           namespace="tenant_id", allow_list=[doc.tenant_id]),
                       IndexDatapoint.Restriction(
                           namespace="kind", allow_list=[c.get("kind", "text")]),
                       # The ledger (12.5): a new version is current; retire_previous removes the old
                       # ids, and the query-time restrict is what a reader asks for.
                       IndexDatapoint.Restriction(namespace="current", allow_list=["true"])],
        )
        for i, (c, v) in enumerate(zip(chunks, vectors))
    ]


def upsert(index_name: str, datapoints: list[IndexDatapoint]) -> None:
    if DRY_RUN:
        print(f"  [dry-run] would upsert {len(datapoints)} datapoints, "
              f"ids {datapoints[0].datapoint_id} .. {datapoints[-1].datapoint_id}")
        return
    # Streaming upserts need an index created with STREAM_UPDATE. A BATCH_UPDATE
    # index accepts the call and applies nothing until the next batch job, which
    # looks exactly like a slow index.
    aiplatform.MatchingEngineIndex(index_name).upsert_datapoints(
        datapoints=datapoints)


def remove_datapoints(index_name: str, ids: list[str]) -> None:
    """The full profile's half of retiring a version: the old ids leave the ANN tier (permanent there - the
    Firestore rows keep the flag and the history)."""
    if not ids:
        return
    if DRY_RUN:
        print(f"  [dry-run] would remove {len(ids)} datapoints, ids {ids[0]} .. {ids[-1]}")
        return
    aiplatform.MatchingEngineIndex(index_name).remove_datapoints(datapoint_ids=ids)


def mirror_to_firestore(db: firestore.Client, doc, chunks: list[dict],
                        vectors: list[list[float]]) -> None:
    """The payload store, and the chaos fallback.

    Vector Search holds the vectors; Firestore holds the text the model quotes.
    Storing the embedding here TOO, as a Vector field, means find_nearest() can
    answer while Vector Search is unavailable - slower and good enough, instead
    of an outage.

    The document is THE canonical shape (2.3, 4.2, 4.5, rag-api): tenant_id, text,
    source_uri, page_start, doc_type, embedding - plus, for a figure or a video
    segment, kind / media_url / start / end. resolve() in shared/documind_schemas.py
    reads exactly these names into a Citation, so what is written here is what the
    frontend renders as a thumbnail or a timestamp (gap G7). A text chunk carries
    kind="text" and nothing else new, so nothing written before Module 9 changes.
    """
    batch = db.batch()
    for i, (c, vec) in enumerate(zip(chunks, vectors)):
        ref = db.collection("chunks").document(doc.chunk_id(i))
        row = {"tenant_id": doc.tenant_id, "text": c["text"],
               "source_uri": doc.gcs_uri, "page_start": c.get("page_start"),
               "doc_type": doc.doc_type, "kind": c.get("kind", "text"),
               # The ledger (11 September 2026): which version this chunk belongs to, that it is the
               # current one, and when it landed. retire_previous() flips `current` on the predecessor.
               "doc_key": doc.doc_key, "current": True,
               "indexed_at": firestore.SERVER_TIMESTAMP,
               "embedding": Vector(vec)}
        if getattr(doc, "effective_from", None):
            row["effective_from"] = doc.effective_from
        for k in ("media_url", "start", "end"):
            if c.get(k) is not None:
                row[k] = c[k]
        batch.set(ref, row)
    batch.commit()


def mirror_to_bigquery(bq, table: str, doc, chunks: list[dict], pii_chunk_ids: set) -> int:
    """The SQL lane's copy of the REAL chunks (lesson 5.5, gap G9).

    One row per chunk into rag_data.chunk_source - the same canonical names, plus the DLP
    verdict this worker already computed, so 5.5's feature job needs no second scan and no
    BigQuery model to know pii_flag. insertId = chunk id: a retried message re-inserts the
    same rows and BigQuery de-duplicates them. `table` is BQ_CHUNK_TABLE
    (PROJECT.rag_data.chunk_source); unset means the lane is off and nothing is written.
    """
    if not table:
        return 0
    from datetime import datetime, timezone

    now = datetime.now(timezone.utc).isoformat()
    rows = [{"chunk_id": doc.chunk_id(i), "tenant_id": doc.tenant_id, "text": c["text"],
             "source_uri": doc.gcs_uri, "page_start": c.get("page_start"), "page_end": None,
             "doc_type": doc.doc_type, "kind": c.get("kind", "text"), "heading_path": None,
             "last_revised_at": None, "pii_flag": doc.chunk_id(i) in pii_chunk_ids,
             "ingested_at": now} for i, c in enumerate(chunks)]
    errors = bq.insert_rows_json(table, rows, row_ids=[r["chunk_id"] for r in rows])
    if errors:
        raise RuntimeError(f"BigQuery rejected {len(errors)} chunk_source row(s): {errors[0]}")
    return len(rows)
'''

with open('indexer.py', 'w') as f: f.write(INDEXER_PY)
print('wrote indexer.py')


In [ ]:
WORKER_PY = '''
"""The Cloud Run worker behind a Pub/Sub push subscription."""
import base64
import json
import logging
import os
import sys

from fastapi import FastAPI, HTTPException, Request
from google import genai
from google.genai import types as gtypes
from google.cloud import firestore, storage
from pydantic import BaseModel, ValidationError

# shared/ ships beside the service in the image (see the Dockerfile), the same
# way services/chat consumes documind_tools.
from shared.pii import inspect_image as pii_inspect_image, inspect_many as pii_inspect_many
from shared.audit_log import emit as audit_emit

from contracts import IngestMessage, DocumentContract, effective_from_of, sha256_of
from idempotency import (claim, drop_tenant_cache, finish, reactivate, record_source, release,
                         retire_previous, status_of)
from indexer import (embed_all, mirror_to_bigquery, mirror_to_firestore, remove_datapoints,
                     to_datapoints, upsert)
from parser import parse

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout)
log = logging.getLogger("documind.ingest")
app = FastAPI()
_db = firestore.Client()
_gcs = storage.Client()
INDEX_NAME = os.environ.get("VECTOR_INDEX_NAME", "")
PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
PROCESSOR_ID = os.environ.get("DOCAI_PROCESSOR_ID", "")    # docai.tf outputs it
GEN_MODEL = os.environ.get("GEN_MODEL", "gemini-3.6-flash")
# The SQL lane (5.5, gap G9): PROJECT.rag_data.chunk_source, declared by dataplex.tf. Unset
# means the lane is off; the worker never needs BigQuery to index a document.
BQ_CHUNK_TABLE = os.environ.get("BQ_CHUNK_TABLE", "")
_bq = None


def _bigquery():
    global _bq
    if _bq is None:
        from google.cloud import bigquery
        _bq = bigquery.Client(project=PROJECT)
    return _bq

# Fail at STARTUP, not on the first document. audit_log.emit refuses to drop an
# event, so an unset AUDIT_BUCKET would surface as a 500 halfway through an
# ingest, release the claim and retry into the DLQ - a configuration mistake
# wearing the costume of a data problem.
if not os.environ.get("AUDIT_BUCKET"):
    raise RuntimeError(
        "AUDIT_BUCKET is not set. The ingest worker records doc.upload and "
        "dlp.finding events; refusing to start without somewhere to put them.")
# parser.py sends a PDF to Document AI in 15-page slices at roughly a second a page, and the
# push subscription allows 600 seconds before it redelivers, so a document this size finishes
# inline with room to spare. Bigger than this goes to the batch lane - a claim document that
# nothing polls yet: the largest file in the kit's corpus is under two hundred pages.
MAX_INLINE_PAGES = 250

# 4.1's chunker, the shape every lesson's corpus has: ~500 tokens per chunk, an
# overlap so a sentence is never cut in half between two chunks.
CHUNK_CHARS, CHUNK_OVERLAP = 2000, 200

# The corpus has four modalities (Module 9) and ONE contract. An uploaded image, video or
# audio file is not parsed for text - it is DESCRIBED, and the description is what gets
# embedded and quoted; the asset rides alongside as media_url (9.6). kind names are
# the shared contract's: figure, table, segment - never a second vocabulary. The key is the
# content type the object.finalized record carries, which `gcloud storage cp`, the UI's
# uploader and 9.4's signed PUT all set from the file - not the extension.
MEDIA_TYPES = {"image/png": "figure", "image/jpeg": "figure",
               "video/mp4": "segment", "audio/mpeg": "segment"}
_gen = None


def _genai() -> genai.Client:
    # Generation is global-only (Gemini 3.x); lazy, so the worker starts without it.
    global _gen
    if _gen is None:
        _gen = genai.Client(enterprise=True, project=PROJECT, location="global")
    return _gen


def _parse(content: bytes, content_type: str) -> tuple[str, int]:
    """(text, pages). Plain text needs no processor; everything else goes to Doc AI."""
    if content_type.startswith("text/"):
        text = content.decode("utf-8", "replace")
        return text, max(1, text.count("\\f") + 1)
    if not PROCESSOR_ID:
        raise RuntimeError("DOCAI_PROCESSOR_ID is not set (deploy/terraform/docai.tf outputs it)")
    return parse(PROJECT, PROCESSOR_ID, content, content_type)


def _chunk(text: str) -> list[dict]:
    """Fixed windows with overlap, ending on a sentence when there is one nearby.

    A text upload marks its page breaks with a form feed (the kit's real-document mirrors,
    evals/fetch_real.py, do; so does _parse's page count), and a chunk that starts on page 7
    is cited as page 7 - the same page_start shared/documind_corpus.py mints for the same
    bytes in a notebook. Text without form feeds has no page to name, and None is more honest
    than 1."""
    text = text.strip()
    paged = "\\f" in text
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + CHUNK_CHARS)
        if end < len(text):
            cut = text.rfind(". ", start + CHUNK_CHARS // 2, end)
            if cut != -1:
                end = cut + 1
        piece = text[start:end].strip()
        if piece:
            out.append({"text": piece, "kind": "text",
                        "page_start": text.count("\\f", 0, start) + 1 if paged else None})
        if end >= len(text):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)
    return out


class Segment(BaseModel):
    start: float
    end: float
    summary: str


def _describe_media(gcs_uri: str, content_type: str) -> list[dict]:
    """9.6: a figure cannot be retrieved as pixels, so the caption IS the retrievable
    body; a video becomes segments with start/end in seconds. from_uri: the asset stays
    in GCS and never passes through this process."""
    kind = MEDIA_TYPES[content_type]
    part = gtypes.Part.from_uri(file_uri=gcs_uri, mime_type=content_type)
    if kind == "figure":
        r = _genai().models.generate_content(
            model=GEN_MODEL,
            contents=[part, "Describe this figure for retrieval: one caption sentence, then the "
                            "key facts it shows, then any table it contains as Markdown."],
            config=gtypes.GenerateContentConfig(
                thinking_config=gtypes.ThinkingConfig(thinking_level="LOW")))
        return [{"text": (r.text or "").strip(), "kind": "figure", "media_url": gcs_uri}]
    audio = content_type.startswith("audio/")
    what = "recording" if audio else "video"
    shown = "what is said" if audio else "what is said and shown"
    config = dict(response_mime_type="application/json", response_schema=list[Segment],
                  thinking_config=gtypes.ThinkingConfig(thinking_level="LOW"))
    if not audio:
        # LOW: 'what was said and roughly when' does not need to read text off slides (9.4).
        # A resolution is a frame-sampling dial; an audio file has no frames to sample.
        config["media_resolution"] = gtypes.MediaResolution.MEDIA_RESOLUTION_LOW
    # "quoting every number": a summary paraphrases, and the first live town hall came back as "a slight
    # contraction" where the speaker said "fell 5.2 per cent" - the segment was found, the figure was
    # gone. The caption is the quote (9.6): what is not in the segment's text cannot be retrieved by it.
    r = _genai().models.generate_content(
        model=GEN_MODEL,
        contents=[part, f"Split this {what} into segments of at most 60 seconds. For each, give start "
                        f"and end in seconds and a two-sentence summary of {shown}, quoting every number, "
                        f"percentage, amount and name that is spoken exactly as it is said."],
        config=gtypes.GenerateContentConfig(**config))
    return [{"text": s.summary, "kind": "segment", "media_url": gcs_uri,
             "start": s.start, "end": s.end} for s in (r.parsed or [])]


def _enqueue_batch(doc: DocumentContract) -> None:
    """The batch lane. A claim document the batch worker polls; no second queue to provision."""
    _db.collection("ingest_batch").document(doc.doc_key).set({
        "tenant_id": doc.tenant_id, "gcs_uri": doc.gcs_uri, "pages": doc.pages,
        "status": "queued", "queued_at": firestore.SERVER_TIMESTAMP})


@app.post("/")
async def push(request: Request):
    envelope = await request.json()
    try:
        raw = base64.b64decode(envelope["message"]["data"])
        msg = IngestMessage.model_validate_json(raw)
    except (KeyError, ValueError, ValidationError) as e:
        # 400, NOT 500. A message this worker can never parse must not be
        # retried five times before reaching the DLQ - it will fail the same
        # way every time. Ack the poison and let the DLQ hold it.
        log.warning(json.dumps({"event": "ingest_poison", "error": str(e)[:200]}))
        raise HTTPException(400, "unparseable message")

    blob = _gcs.bucket(msg.bucket).blob(msg.name)
    content = blob.download_as_bytes()
    doc = DocumentContract(tenant_id=msg.tenant_id, sha256=sha256_of(content),
                           gcs_uri=msg.gcs_uri, pages=0)

    if not claim(_db, doc.doc_key, doc.gcs_uri):
        if status_of(_db, doc.doc_key) == "superseded":
            # THE UNDO (the ledger, 11 September 2026). The same bytes again, after a newer version
            # retired them: the chunks are still here, flagged. Flip them back, retire the newer
            # version in turn, and nothing is re-embedded - because nothing was ever deleted.
            back = reactivate(_db, doc.tenant_id, doc.gcs_uri, doc.doc_key)
            gone = retire_previous(_db, doc.tenant_id, doc.gcs_uri, doc.doc_key)
            if INDEX_NAME and gone["retired_ids"]:
                remove_datapoints(INDEX_NAME, gone["retired_ids"])
            record_source(_db, doc.tenant_id, msg.name, doc.gcs_uri, doc.doc_key, msg.generation,
                          doc.sha256, back, effective_from_of(msg.name, None))
            drop_tenant_cache(_db, doc.tenant_id)
            log.info(json.dumps({"event": "ingest_reactivated", "tenant": doc.tenant_id,
                                 "doc_key": doc.doc_key, "chunks": back,
                                 "retired": gone["retired_doc_keys"]}))
            return {"status": "reactivated", "doc_key": doc.doc_key, "chunks": back}
        # Already done by an earlier delivery, or by an earlier upload of the
        # same bytes. Returning 200 ACKS the message: this is a success, not a
        # failure, and retrying it would achieve nothing.
        return {"status": "duplicate", "doc_key": doc.doc_key}

    gone = {"retired_doc_keys": [], "retired_ids": [], "retired_chunks": 0}
    try:
        image_findings = []
        if msg.content_type in MEDIA_TYPES:
            chunks = _describe_media(doc.gcs_uri, msg.content_type)
            pages = 1
            doc = doc.model_copy(update={"pages": 1, "doc_type": MEDIA_TYPES[msg.content_type],
                                        "effective_from": effective_from_of(msg.name, None)})
            # 9.6: the PIXELS are scanned, not only the caption. The caption is Gemini's
            # description of the picture, and a description of an invoice can carry the
            # invoice's PAN in plain text - so the scan below would find it there too, but a
            # picture of a form with a PAN the caption did not mention would sail into the
            # index. Same info-types, same no-quote rule (shared/pii.py); a video is not an
            # image DLP can read and yields nothing here.
            image_findings = pii_inspect_image(content, msg.content_type)
        else:
            text, pages = _parse(content, msg.content_type)
            doc = doc.model_copy(update={"pages": pages,
                                        "effective_from": effective_from_of(msg.name, text)})
            if pages > MAX_INLINE_PAGES:
                # A 400-page contract will not finish inside a push request's
                # timeout. Hand it to the batch lane and ack.
                _enqueue_batch(doc)
                return {"status": "queued_batch", "pages": pages}
            chunks = _chunk(text)

        # Scan BEFORE indexing. After the upsert the PII is in the index, and
        # "we scanned it afterwards" is a description of a breach, not a control.
        # A DLP failure fails the whole message: it is nacked, retried, and ends
        # in the DLQ where a human decides - because indexing an unscanned
        # document is the exact thing this control exists to prevent.
        # One scan per document, not per chunk: DLP meters requests per minute, and a
        # corpus load from ten workers at a chunk a request was refused (first live load).
        findings = [{**f, "chunk_id": doc.chunk_id(0)} for f in image_findings]
        for i, chunk_findings in enumerate(pii_inspect_many([c["text"] for c in chunks])):
            for f in chunk_findings:
                findings.append({**f, "chunk_id": doc.chunk_id(i)})
        if findings:
            # No quotes, only types and offsets - see shared/pii.py.
            _db.collection("dlp_findings").add({
                "doc_key": doc.doc_key, "tenant_id": doc.tenant_id,
                "findings": findings, "count": len(findings),
                "scanned_at": firestore.SERVER_TIMESTAMP,
            })
            audit_emit("dlp.finding",
                       actor={"tenant_id": doc.tenant_id, "email": "system:ingest"},
                       target={"type": "document", "id": doc.doc_key,
                               "tenant_id": doc.tenant_id},
                       meta={"types": sorted({f["info_type"] for f in findings}),
                             "count": len(findings)})

        vectors = embed_all([c["text"] for c in chunks])
        if INDEX_NAME:                       # the full profile; the lean one has no index
            upsert(INDEX_NAME, to_datapoints(doc, chunks, vectors))
        mirror_to_firestore(_db, doc, chunks, vectors)
        # THE LEDGER (11 September 2026). The new version is current from the line above; every
        # other version of this object path is now retired - flagged, never deleted - so the index
        # holds exactly one current reading of a document. A re-issued handbook replaces its
        # predecessor instead of standing beside it, and a citation opens the page it quotes.
        # After the write, never before: a reader between the two steps still finds a document.
        gone = retire_previous(_db, doc.tenant_id, doc.gcs_uri, doc.doc_key)
        if INDEX_NAME and gone["retired_ids"]:
            remove_datapoints(INDEX_NAME, gone["retired_ids"])
        # The SQL lane reads the REAL chunks (5.5, gap G9): the same rows, with the verdict
        # the scan above just produced, so pii_flag in BigQuery is this worker's - never a
        # second scanner's that could disagree.
        mirror_to_bigquery(_bigquery() if BQ_CHUNK_TABLE else None, BQ_CHUNK_TABLE, doc, chunks,
                           {f["chunk_id"] for f in findings})
        finish(_db, doc.doc_key, len(chunks))
        record_source(_db, doc.tenant_id, msg.name, doc.gcs_uri, doc.doc_key, msg.generation,
                      doc.sha256, len(chunks), doc.effective_from)
        # The cache follows the index: the tenant's pack record goes, the next answer runs uncached,
        # make cache rebuilds the pack from the corpus that changed (12.6, 10.2).
        drop_tenant_cache(_db, doc.tenant_id)
        if gone["retired_chunks"]:
            log.info(json.dumps({"event": "ingest_superseded", "tenant": doc.tenant_id,
                                 "doc_key": doc.doc_key, "gcs_uri": doc.gcs_uri,
                                 "retired_doc_keys": gone["retired_doc_keys"],
                                 "retired_chunks": gone["retired_chunks"]}))

        # The document is now retrievable. Record that, with who and what - the
        # upload event the audit trail is missing without it.
        audit_emit("doc.upload",
                   actor={"tenant_id": doc.tenant_id, "email": "system:ingest"},
                   target={"type": "document", "id": doc.doc_key,
                           "tenant_id": doc.tenant_id},
                   meta={"gcs_uri": doc.gcs_uri, "pages": doc.pages,
                         "chunks": len(chunks), "pii": bool(findings),
                         "kinds": sorted({c["kind"] for c in chunks})})
    except Exception as e:
        # Give the claim back before failing, or the retry finds the document
        # already claimed and does nothing - for ever. And SAY what failed, on the log
        # line an operator reads first: the claim document carries the same text, but
        # the first live load was diagnosed from request logs that only said 500.
        release(_db, doc.doc_key, f"{type(e).__name__}: {e}")
        log.error(json.dumps({"event": "ingest_failed", "tenant": doc.tenant_id,
                              "doc_key": doc.doc_key, "gcs_uri": doc.gcs_uri,
                              "error": f"{type(e).__name__}: {e}"[:600]}))
        raise HTTPException(500, "ingest failed")

    log.info(json.dumps({"event": "ingest_ok", "tenant": doc.tenant_id,
                         "doc_key": doc.doc_key, "chunks": len(chunks),
                         "pages": pages, "kinds": sorted({c["kind"] for c in chunks}),
                         "retired": gone["retired_chunks"], "effective_from": doc.effective_from}))
    return {"status": "indexed", "chunks": len(chunks)}
'''

with open('main.py', 'w') as f: f.write(WORKER_PY)
print('wrote main.py')


In [ ]:
VECTOR_TF = '''
# The STREAM_UPDATE index, provisioned for the first time in this lesson.
# Modules 4 and 12.2 have been querying an index that the course never created -
# this is it.
resource "google_vertex_ai_index" "documind" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  region       = var.region
  display_name = "documind-chunks"
  description  = "DocuMind chunk embeddings, 768-d, streaming upserts"

  metadata {
    contents_delta_uri = "gs://${google_storage_bucket.uploads.name}/index-delta"
    config {
      dimensions                  = 768
      approximate_neighbors_count = 150
      distance_measure_type       = "DOT_PRODUCT_DISTANCE"
      algorithm_config {
        tree_ah_config {
          leaf_node_embedding_count    = 500
          leaf_nodes_to_search_percent = 7
        }
      }
    }
  }

  # STREAM_UPDATE, not BATCH_UPDATE. With BATCH_UPDATE the worker's
  # upsert_datapoints call is accepted and applied at the next batch job, so an
  # uploaded document is simply absent for hours and nothing reports an error.
  index_update_method = "STREAM_UPDATE"
}

resource "google_vertex_ai_index_endpoint" "documind" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  region                  = var.region
  display_name            = "documind-endpoint"
  public_endpoint_enabled = true
}

# An index and an endpoint are two things; neither of them serves a query. The
# DEPLOYED index is the third, and it is the one that costs money per hour - which
# is why it is easy to leave out of the terraform and then wonder why
# find_neighbors returns nothing against an index that plainly exists.
resource "google_vertex_ai_index_endpoint_deployed_index" "documind" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  index_endpoint    = google_vertex_ai_index_endpoint.documind[0].id
  index             = google_vertex_ai_index.documind[0].id
  deployed_index_id = "documind_chunks_v1"
  display_name      = "documind-chunks-v1"

  # One small replica. This is the line to raise for a live cohort and the line
  # to drop to zero afterwards - see the scale-down runbook in deploy/README.
  dedicated_resources {
    machine_spec { machine_type = "e2-standard-2" }
    min_replica_count = 1
    max_replica_count = 1
  }
}

# These two outputs are the whole contract with rag-api/config.py, which reads
# them as VECTOR_INDEX_ENDPOINT and VECTOR_DEPLOYED_INDEX_ID. The endpoint output
# is the RESOURCE NAME, not the public domain: retriever.py passes it straight to
# aiplatform.MatchingEngineIndexEndpoint(), which wants the name.
output "vector_index_endpoint" {
  description = "VECTOR_INDEX_ENDPOINT for rag-api"
  value       = one(google_vertex_ai_index_endpoint.documind[*].id)
}

output "vector_deployed_index_id" {
  description = "VECTOR_DEPLOYED_INDEX_ID for rag-api"
  value       = one(google_vertex_ai_index_endpoint_deployed_index.documind[*].deployed_index_id)
}

output "vector_index_name" {
  description = "VECTOR_INDEX_NAME for the ingest worker's upsert_datapoints"
  value       = one(google_vertex_ai_index.documind[*].id)
}
'''

with open('vector.tf', 'w') as f: f.write(VECTOR_TF)
print('wrote vector.tf')


In [ ]:
DOCAI_TF = '''
# One processor, chosen by residency - the terraform half of parser.py's PROCESSORS map.
# Keep the two in step: a type added here without a row there is a processor nothing calls.
locals {
  docai = {
    india = { location = "asia-south1", type = "OCR_PROCESSOR" }
    us    = { location = "us", type = "LAYOUT_PARSER_PROCESSOR" }
  }
  docai_cfg = local.docai[var.residency]
}

resource "google_document_ai_processor" "documind" {
  project      = var.project_id
  location     = local.docai_cfg.location
  display_name = "documind-parser"
  type         = local.docai_cfg.type
}

# parser.py builds the processor path itself, so it needs the bare id and the
# location separately - not the full resource name.
output "docai_processor_id" {
  description = "DOCAI_PROCESSOR_ID for the ingest worker"
  value       = element(split("/", google_document_ai_processor.documind.id), 5)
}

output "docai_location" {
  description = "matches RESIDENCY in services/ingest/parser.py"
  value       = local.docai_cfg.location
}
'''

with open('docai.tf', 'w') as f: f.write(DOCAI_TF)
print('wrote docai.tf')
print()
print("residency -> processor, as terraform will resolve it:")
for res, cfg in {"india": ("asia-south1", "OCR_PROCESSOR"),
                 "us": ("us", "LAYOUT_PARSER_PROCESSOR")}.items():
    print(f"  var.residency = {res:6} -> {cfg[1]:24} in {cfg[0]}")
print()
print("The Layout Parser gives structure - headings, tables, reading order - and runs in")
print("`us` only. An India-resident tenant gets Enterprise OCR, which returns text and")
print("positions but no layout. 12.5's chunker has to cope with both.")


In [ ]:
FIRESTORE_INDEXES_TF = '''
# Two vector indexes, both 768-d to match text-embedding-005.

# chunks: the chaos fallback for retriever.py. Written by indexer.py on every
# ingest; queried only when Vector Search is down.
resource "google_firestore_index" "chunks_vector" {
  project     = var.project_id
  database    = google_firestore_database.main.name
  collection  = "chunks"
  query_scope = "COLLECTION"

  # Equality filter FIRST, vector field LAST. Firestore will not accept the
  # reverse, and the error message points at an index name that looks correct.
  fields {
    field_path = "tenant_id"
    order      = "ASCENDING"
  }
  # Firestore records the document key between the ordered fields and the vector field, and
  # reports it back in that position. Declared here the same way, the provider reads the
  # index it wrote. Left out, every apply read a three-field index against a two-field
  # definition, planned a replacement, and the asynchronous deletion raced the re-creation
  # into a 409 - the first live applies (the first live run, 6 September 2026) looped on exactly that.
  fields {
    field_path = "__name__"
    order      = "ASCENDING"
  }
  fields {
    field_path = "embedding"
    vector_config {
      dimension = 768
      flat {}
    }
  }
}

# chunks, current only: the ledger's promise (12.5, 11 September 2026). A SECOND index, not a change to the
# first - a changed vector index is destroyed and re-created, and retrieval would be refused while it builds;
# this one builds beside the first, and RETRIEVAL_CURRENT_ONLY=on on the API is what starts using it, after
# `make backfill-current` has stamped the chunks written before the ledger.
resource "google_firestore_index" "chunks_current_vector" {
  project     = var.project_id
  database    = google_firestore_database.main.name
  collection  = "chunks"
  query_scope = "COLLECTION"

  fields {
    field_path = "tenant_id"
    order      = "ASCENDING"
  }
  fields {
    field_path = "current"
    order      = "ASCENDING"
  }
  fields {
    field_path = "__name__"
    order      = "ASCENDING"
  }
  fields {
    field_path = "embedding"
    vector_config {
      dimension = 768
      flat {}
    }
  }
}

# answer_cache: the semantic cache from 12.6. Same shape, different collection -
# and the tenant_id filter is what stops one customer's answer reaching another.
resource "google_firestore_index" "answer_cache_vector" {
  project     = var.project_id
  database    = google_firestore_database.main.name
  collection  = "answer_cache"
  query_scope = "COLLECTION"

  fields {
    field_path = "tenant_id"
    order      = "ASCENDING"
  }
  # Firestore records the document key between the ordered fields and the vector field, and
  # reports it back in that position. Declared here the same way, the provider reads the
  # index it wrote. Left out, every apply read a three-field index against a two-field
  # definition, planned a replacement, and the asynchronous deletion raced the re-creation
  # into a 409 - the first live applies (the first live run, 6 September 2026) looped on exactly that.
  fields {
    field_path = "__name__"
    order      = "ASCENDING"
  }
  fields {
    field_path = "embedding"
    vector_config {
      dimension = 768
      flat {}
    }
  }
}

# The roster's reverse lookup (shared/tenancy.py tenant_for, lesson 12.8: "which tenant is
# this person on?") is a COLLECTION-GROUP query on members.email, across every tenant's
# members at once. Firestore indexes a single field at collection scope by default and
# refuses a collection-group query on it - the UI's first page was a FailedPrecondition
# on the first live sign-in (the first live run, 6 September 2026). This field override adds the
# collection-group index; the point lookup (is_member) reads by document id and needs none.
resource "google_firestore_field" "members_email" {
  project    = var.project_id
  database   = google_firestore_database.main.name
  collection = "members"
  field      = "email"
  index_config {
    indexes {
      order       = "ASCENDING"
      query_scope = "COLLECTION"
    }
    indexes {
      order       = "ASCENDING"
      query_scope = "COLLECTION_GROUP"
    }
  }
}
'''

with open('firestore_indexes.tf', 'w') as f: f.write(FIRESTORE_INDEXES_TF)
print('wrote firestore_indexes.tf')
print()
print('Index          collection     filter        vector field  dims')
print('-' * 62)
for name, coll in (('chunks_vector', 'chunks'), ('answer_cache_vector', 'answer_cache')):
    print(f'{name:14} {coll:14} tenant_id ==  embedding     768')
print()
print('Both are FLAT indexes. Firestore has no ANN tier to choose here: flat is')
print('exact and the collections are small enough per tenant that exact is fine.')


In [ ]:
EVENTARC_TF = '''
# GCS -> Pub/Sub -> Cloud Run, with a dead-letter topic.
#
# The bucket publishes its own object.finalized records (a Cloud Storage notification,
# payload JSON_API_V1) into the topic below, and the push subscription delivers them to the
# worker with an OIDC token and a floor of five attempts. The worker's IngestMessage is that
# record, field for field. An earlier draft declared an Eventarc trigger here instead: Eventarc
# delivers a CloudEvent straight to the service, around this subscription, its token, its retry
# ceiling and its DLQ - and the worker, which parses a Pub/Sub envelope, answered 400 to it.
data "google_project" "current" {}
data "google_storage_project_service_account" "gcs" {}

resource "google_pubsub_topic" "ingest" { name = "documind-ingest" }
resource "google_pubsub_topic" "ingest_dlq" { name = "documind-ingest-dlq" }

# Cloud Storage publishes as its own service agent. Without this grant the notification is
# created and nothing ever arrives - the first silent failure on this path.
resource "google_pubsub_topic_iam_member" "gcs_publishes" {
  topic  = google_pubsub_topic.ingest.id
  role   = "roles/pubsub.publisher"
  member = "serviceAccount:${data.google_storage_project_service_account.gcs.email_address}"
}

resource "google_storage_notification" "uploads" {
  bucket         = google_storage_bucket.uploads.name
  topic          = google_pubsub_topic.ingest.id
  payload_format = "JSON_API_V1"
  event_types    = ["OBJECT_FINALIZE"]
  depends_on     = [google_pubsub_topic_iam_member.gcs_publishes]
}

locals {
  # Cloud Run's deterministic URL: the service name and the project NUMBER, no hash to look
  # up after the first deploy. commands/lesson-12.2.sh builds SELF_URL the same way.
  ingest_url = "https://documind-ingest-${data.google_project.current.number}.${var.region}.run.app"
}

# Pub/Sub mints the push token AS the ingest service account, which takes this grant to
# Pub/Sub's own service agent - the second silent failure. The dead-letter hop needs the
# same agent to publish to the DLQ topic and to subscribe here.
resource "google_service_account_iam_member" "pubsub_mints_ingest_token" {
  service_account_id = google_service_account.ingest.name
  role               = "roles/iam.serviceAccountTokenCreator"
  member             = "serviceAccount:service-${data.google_project.current.number}@gcp-sa-pubsub.iam.gserviceaccount.com"
}

resource "google_pubsub_topic_iam_member" "dlq_publisher" {
  topic  = google_pubsub_topic.ingest_dlq.id
  role   = "roles/pubsub.publisher"
  member = "serviceAccount:service-${data.google_project.current.number}@gcp-sa-pubsub.iam.gserviceaccount.com"
}

resource "google_pubsub_subscription" "ingest_push" {
  name  = "documind-ingest-push"
  topic = google_pubsub_topic.ingest.name

  push_config {
    push_endpoint = local.ingest_url
    oidc_token {
      # The worker is --no-allow-unauthenticated. Pub/Sub mints an OIDC token
      # for this service account, so the endpoint is reachable by Pub/Sub and
      # by nothing else on the internet. commands/lesson-12.5.sh grants the
      # account run.invoker on the service once the service exists.
      service_account_email = google_service_account.ingest.email
    }
  }

  # At-least-once, with a ceiling: attempts with exponential backoff, then the message goes
  # to the DLQ instead of being retried for a week. Twelve, not five: a corpus upload lands
  # thirty objects at once, the worker takes one request per instance, and Cloud Run refuses
  # ("no available instance") while it cold-starts - the first live load sent a document
  # to the DLQ on five refusals that were never the document's fault.
  retry_policy {
    minimum_backoff = "10s"
    maximum_backoff = "600s"
  }
  dead_letter_policy {
    dead_letter_topic     = google_pubsub_topic.ingest_dlq.id
    max_delivery_attempts = 12
  }
  ack_deadline_seconds = 600
  depends_on           = [google_service_account_iam_member.pubsub_mints_ingest_token]
}

resource "google_pubsub_subscription_iam_member" "dlq_subscriber" {
  subscription = google_pubsub_subscription.ingest_push.name
  role         = "roles/pubsub.subscriber"
  member       = "serviceAccount:service-${data.google_project.current.number}@gcp-sa-pubsub.iam.gserviceaccount.com"
}

# Somewhere to read the poison from. The README's Tier B block pulls from it by name.
resource "google_pubsub_subscription" "ingest_dlq_sub" {
  name  = "ingest-dlq-sub"
  topic = google_pubsub_topic.ingest_dlq.name
}
'''

with open('eventarc.tf', 'w') as f: f.write(EVENTARC_TF)
print('wrote eventarc.tf')


In [ ]:
REQUIREMENTS = '''
fastapi==0.141.1
uvicorn[standard]==0.52.4
gunicorn==26.2.0
pydantic==2.13.5
google-cloud-firestore==2.30.0
google-cloud-storage==3.13.1
google-cloud-documentai==3.15.0
pypdf==6.17.0
google-cloud-aiplatform==1.153.1
google-genai==2.22.0
google-cloud-dlp==3.39.0
# 9.6: shared/pii.py re-encodes a page render under DLP's 0.5 MiB inspect limit before scanning its pixels.
Pillow==12.3.0
# The SQL lane (lesson 5.5, gap G9): one chunk_source row per indexed chunk, when BQ_CHUNK_TABLE is set.
google-cloud-bigquery==3.45.0
'''

with open('requirements.txt', 'w') as f: f.write(REQUIREMENTS)
print('wrote requirements.txt')


In [ ]:
DOCKERFILE = '''
# syntax=docker/dockerfile:1.7
FROM python:3.12-slim
RUN apt-get update && apt-get install -y --no-install-recommends tini ca-certificates && rm -rf /var/lib/apt/lists/*
RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app
COPY services/ingest/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=app:app shared/ ./shared/\nCOPY --chown=app:app services/ingest/ .
USER app
ENV PORT=8080 PYTHONUNBUFFERED=1
EXPOSE 8080
# CONCURRENCY 1, and one worker. The process holds a document claim while it
# parses and embeds; a second request on the same instance contends for CPU and
# stretches the ack window, and Pub/Sub redelivers whatever is not acked in
# time - which is how one slow instance turns one document into three
# deliveries and three claims.
ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["gunicorn", "-k", "uvicorn.workers.UvicornWorker", "-w", "1", "-b", "0.0.0.0:8080", "-t", "600", "--access-logfile", "-", "main:app"]
'''

with open('Dockerfile', 'w') as f: f.write(DOCKERFILE)
print('wrote Dockerfile')


In [ ]:
DEPLOY = '''
# The ingest worker. Reached by nothing but the push subscription in eventarc.tf, which
# delivers as documind-ingest-sa with an OIDC token - hence internal ingress, no
# unauthenticated calls, and the invoker grant to that account below. Concurrency 1 (see the
# Dockerfile), and a 600-second request ceiling to match the subscription's ack deadline:
# a hundred-page Act is a dozen Document AI calls and fits. Thirty instances, so a corpus
# upload of thirty objects is not a queue of cold-start refusals.
#
# RESIDENCY picks the processor the way docai.tf did (us: Layout Parser; india: OCR in Mumbai).
# VECTOR_INDEX_NAME and BQ_CHUNK_TABLE are empty in the lean profile, which switches the
# datapoint upsert and the BigQuery mirror off; the Firestore index is always written.
gcloud run deploy documind-ingest \\
  --image=${REGION:-us-central1}-docker.pkg.dev/$PROJECT/documind/ingest:$GIT_SHA \\
  --region=${REGION:-us-central1} --platform=managed \\
  --no-allow-unauthenticated \\
  --ingress=internal \\
  --memory=2Gi --cpu=2 --concurrency=1 --timeout=600 \\
  --min-instances=0 --max-instances=30 \\
  --execution-environment=gen2 \\
  --service-account=documind-ingest-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-env-vars="^|^GOOGLE_CLOUD_PROJECT=$PROJECT|RESIDENCY=${RESIDENCY:-us}|DOCAI_PROCESSOR_ID=$DOCAI_PROCESSOR_ID|AUDIT_BUCKET=$PROJECT-audit|VECTOR_INDEX_NAME=$VECTOR_INDEX_NAME|BQ_CHUNK_TABLE=$BQ_CHUNK_TABLE"

# Pub/Sub calls the worker AS the ingest service account; the account has to be allowed in.
# The subscription already exists, pointing at this service's deterministic URL.
gcloud run services add-iam-policy-binding documind-ingest \\
  --region=${REGION:-us-central1} --project=$PROJECT \\
  --member="serviceAccount:documind-ingest-sa@$PROJECT.iam.gserviceaccount.com" \\
  --role=roles/run.invoker
'''
print(DEPLOY)
